<a href="https://colab.research.google.com/github/rafiyamo/manga-ocr-translation/blob/main/notebooks/train_translation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/rafiyamo/manga-ocr-translation.git
%cd manga-ocr-translation

Cloning into 'manga-ocr-translation'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 117 (delta 48), reused 77 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 28.05 MiB | 18.15 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/manga-ocr-translation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil, pathlib

drive_zip = "/content/drive/MyDrive/jp_en_splits.zip"
repo_zip  = "/content/manga-ocr-translation/data/parallel/jp_en_splits.zip"

pathlib.Path("/content/manga-ocr-translation/data/parallel").mkdir(parents=True, exist_ok=True)
shutil.copy(drive_zip, repo_zip)


'/content/manga-ocr-translation/data/parallel/jp_en_splits.zip'

In [ ]:
!unzip -o /content/manga-ocr-translation/data/parallel/jp_en_splits.zip \
     -d /content/manga-ocr-translation/data/parallel

Archive:  /content/manga-ocr-translation/data/parallel/jp_en_splits.zip
  inflating: /content/manga-ocr-translation/data/parallel/jp_en_dev.tsv  
  inflating: /content/manga-ocr-translation/data/parallel/jp_en_train.tsv  
  inflating: /content/manga-ocr-translation/data/parallel/jp_en_test.tsv  


In [ ]:
!pip install -r requirements.txt

# Install PyTorch with CUDA support (works on most Colab GPUs)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.9.0+cu126
CUDA available: True
GPU name: Tesla T4


In [ ]:
from src.pipeline import process_page

print("Imports OK.")

Imports OK.


In [ ]:
# ===== Config & basic imports =====

import math
from dataclasses import dataclass
from typing import List, Tuple, Dict

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Some basic hyperparameters (we can tune later)
@dataclass
class TrainConfig:
    max_epochs: int = 10
    batch_size: int = 32
    lr: float = 1e-3
    max_src_len: int = 64
    max_tgt_len: int = 64
    embed_dim: int = 128
    hidden_dim: int = 256

config = TrainConfig()
config

Using device: cuda


TrainConfig(max_epochs=10, batch_size=32, lr=0.001, max_src_len=64, max_tgt_len=64, embed_dim=128, hidden_dim=256)

In [ ]:
# === Load real JP–EN corpus from TSV ===
from typing import NamedTuple, List
from pathlib import Path
import csv

class ParallelExample(NamedTuple):
    src: str  # source sentence (Japanese)
    tgt: str  # target sentence (English)

def load_parallel_data(split: str, max_pairs: int | None = None) -> List[ParallelExample]:
    """
    Load JP–EN parallel data from TSV files:
      data/parallel/jp_en_train.tsv
      data/parallel/jp_en_dev.tsv
      data/parallel/jp_en_test.tsv

    Each line is expected to be:
        <jp sentence>\t<en sentence>

    We'll optionally skip a header line if it looks like one.
    """
    path = Path("data/parallel") / f"jp_en_{split}.tsv"
    examples: List[ParallelExample] = []

    with path.open(encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")

        # Peek at the first row to detect a header
        first_row = next(reader, None)
        if first_row is None:
            return examples  # empty file

        def looks_like_header(row: list[str]) -> bool:
            if len(row) != 2:
                return False
            joined = " ".join(c.lower() for c in row)
            return any(k in joined for k in ["src", "tgt", "jp", "ja", "en", "english"])

        # If the first row looks like a header, start from the next row
        if not looks_like_header(first_row):
            # treat it as data
            if len(first_row) >= 2:
                jp = first_row[0].strip()
                en = first_row[1].strip()
                if jp and en:
                    examples.append(ParallelExample(src=jp, tgt=en))

        # Now read the rest
        for row in reader:
            if len(row) < 2:
                continue
            jp = row[0].strip()
            en = row[1].strip()
            if not jp or not en:
                continue

            examples.append(ParallelExample(src=jp, tgt=en))

            if max_pairs is not None and len(examples) >= max_pairs:
                break

    return examples

In [ ]:
train_data = load_parallel_data("train", max_pairs=200_000)
dev_data   = load_parallel_data("dev",   max_pairs=20_000)

len(train_data), len(dev_data), train_data[0]


(200000, 20000, ParallelExample(src='私は また払った', tgt='i paid again.'))

In [ ]:
from typing import Dict, List
from torch.utils.data import Dataset, DataLoader

# Special tokens (must match translate.py)
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"

def build_char_vocab(pairs: List[ParallelExample]) -> Dict[str, int]:
    """Build a character-level vocab from JP and EN text."""
    chars = set()
    for ex in pairs:
        chars.update(list(ex.src))
        chars.update(list(ex.tgt))

    vocab = {
        PAD_TOKEN: 0,
        SOS_TOKEN: 1,
        EOS_TOKEN: 2,
    }
    for ch in sorted(chars):
        if ch not in vocab:
            vocab[ch] = len(vocab)

    return vocab


def encode_text(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    """Encode string as list of ids, with SOS/EOS and padding."""
    ids = [vocab[SOS_TOKEN]]
    for ch in text:
        ids.append(vocab.get(ch, vocab[PAD_TOKEN]))
        if len(ids) >= max_len - 1:
            break
    ids.append(vocab[EOS_TOKEN])

    if len(ids) < max_len:
        ids.extend([vocab[PAD_TOKEN]] * (max_len - len(ids)))
    else:
        ids = ids[:max_len]

    return ids


def decode_ids(ids: List[int], inv_vocab: Dict[int, str]) -> str:
    """Back from ids to string, skipping special tokens."""
    chars = []
    for idx in ids:
        tok = inv_vocab.get(idx, "")
        if tok in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN):
            continue
        chars.append(tok)
    return "".join(chars)


class ParallelCharDataset(Dataset):
    def __init__(self, examples: List[ParallelExample], vocab: Dict[str, int], config: TrainConfig):
        self.examples = examples
        self.vocab = vocab
        self.config = config

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        src_ids = encode_text(ex.src, self.vocab, self.config.max_src_len)
        tgt_ids = encode_text(ex.tgt, self.vocab, self.config.max_tgt_len)
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


# Build vocab on *train* data
vocab = build_char_vocab(train_data)
inv_vocab = {i: tok for tok, i in vocab.items()}
vocab_size = len(vocab)
vocab_size


4220

In [ ]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src_ids):
        emb = self.embedding(src_ids)
        outputs, hidden = self.rnn(emb)
        return hidden  # [1, batch, hidden]


class Decoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_tok, hidden):
        # input_tok: [batch] int ids
        emb = self.embedding(input_tok.unsqueeze(1))  # [batch,1,embed]
        output, hidden = self.rnn(emb, hidden)        # [batch,1,hid]
        logits = self.out(output.squeeze(1))          # [batch, vocab]
        return logits, hidden


class Seq2SeqModel(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim)

    def forward(self, src_ids, tgt_ids, teacher_forcing_ratio=0.5):
        """
        src_ids: [batch, src_len]
        tgt_ids: [batch, tgt_len]
        """
        batch_size, tgt_len = tgt_ids.size()
        vocab_size = self.decoder.out.out_features

        hidden = self.encoder(src_ids)
        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=src_ids.device)

        input_tok = tgt_ids[:, 0]  # <sos>

        for t in range(1, tgt_len):
            logits, hidden = self.decoder(input_tok, hidden)
            outputs[:, t, :] = logits

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = logits.argmax(dim=-1)

            input_tok = tgt_ids[:, t] if teacher_force else top1

        return outputs


In [ ]:
from torch.utils.data import DataLoader

config.max_src_len = 64
config.max_tgt_len = 64
config.max_epochs = 15
config.batch_size = 128
config.embed_dim = 256
config.hidden_dim = 512

train_dataset = ParallelCharDataset(train_data, vocab, config)
dev_dataset   = ParallelCharDataset(dev_data,   vocab, config)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size,
                          shuffle=True, drop_last=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=config.batch_size,
                          shuffle=False, drop_last=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Seq2SeqModel(vocab_size, config.embed_dim, config.hidden_dim).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab[PAD_TOKEN])
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)


def run_epoch(model, loader, train: bool) -> float:
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    num_batches = 0

    for batch_src, batch_tgt in loader:
        batch_src = batch_src.to(device)
        batch_tgt = batch_tgt.to(device)

        if train:
            optimizer.zero_grad()

        outputs = model(batch_src, batch_tgt, teacher_forcing_ratio=0.5 if train else 0.0)
        # shift targets to skip SOS
        logits = outputs[:, 1:, :].reshape(-1, vocab_size)
        targets = batch_tgt[:, 1:].reshape(-1)

        loss = criterion(logits, targets)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)


for epoch in range(1, config.max_epochs + 1):
    train_loss = run_epoch(model, train_loader, train=True)
    dev_loss   = run_epoch(model, dev_loader,   train=False)
    print(f"Epoch {epoch}: train loss={train_loss:.4f}, dev loss={dev_loss:.4f}")


Epoch 1: train loss=2.6331, dev loss=3.1114
Epoch 2: train loss=2.4925, dev loss=3.2144
Epoch 3: train loss=2.4526, dev loss=3.1486
Epoch 4: train loss=2.4219, dev loss=3.1921
Epoch 5: train loss=2.3964, dev loss=3.0886
Epoch 6: train loss=2.3797, dev loss=3.1274
Epoch 7: train loss=2.3606, dev loss=3.2097
Epoch 8: train loss=2.3491, dev loss=3.1695
Epoch 9: train loss=2.3343, dev loss=3.2207
Epoch 10: train loss=2.3206, dev loss=3.1941
Epoch 11: train loss=2.3154, dev loss=3.1807
Epoch 12: train loss=2.3058, dev loss=3.1999
Epoch 13: train loss=2.3032, dev loss=3.2012
Epoch 14: train loss=2.2910, dev loss=3.2124
Epoch 15: train loss=2.2925, dev loss=3.1794


In [ ]:
def translate_text_with_model(model, text: str, max_len: int = None) -> str:
    model.eval()
    if max_len is None:
        max_len = config.max_tgt_len

    src_ids = encode_text(text, vocab, config.max_src_len)
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)

    hidden = model.encoder(src_tensor)

    eos_idx = vocab[EOS_TOKEN]
    input_tok = torch.tensor([vocab[SOS_TOKEN]], dtype=torch.long, device=device)
    decoded_ids = []

    for _ in range(max_len - 1):
        logits, hidden = model.decoder(input_tok, hidden)
        next_tok = logits.argmax(dim=-1)  # [1]
        token_id = next_tok.item()
        if token_id == eos_idx:
            break
        decoded_ids.append(token_id)
        input_tok = next_tok

    inv_vocab = {i: tok for tok, i in vocab.items()}
    return decode_ids(decoded_ids, inv_vocab)

# Quick sanity check
print(translate_text_with_model(model, "私は まだ払った"))


i was a  oo .


In [ ]:
from pathlib import Path

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

save_path = models_dir / "translator.pt"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "vocab": vocab,
    "pad_idx": vocab[PAD_TOKEN],
    "sos_idx": vocab[SOS_TOKEN],
    "eos_idx": vocab[EOS_TOKEN],
    "config": config.__dict__,
}

torch.save(checkpoint, save_path)
print("Saved model to:", save_path)


Saved model to: models/translator.pt


In [ ]:
from google.colab import files
files.download("models/translator.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>